# Abortion Recovery: What Helps? -- r/abortion

**Abstract.** This notebook analyzes 522 treatment reports from 205 users in r/abortion to identify which recovery and comfort measures receive the most positive sentiment. After filtering out procedure names (mifepristone, misoprostol, etc.), we focus on what people use *after* the procedure to manage pain, nausea, and discomfort. Ibuprofen (18 users, 67% positive), heating pads (10 users, 70% positive), and tylenol (7 users, 100% positive) emerge as the most-discussed recovery aids, all with strongly positive sentiment. Tylenol is the only recovery treatment whose positive rate reaches statistical significance via binomial test (p = 0.016); zofran approaches significance (5 users, 100% positive, p = 0.063). The data suggest a simple, accessible recovery toolkit: ibuprofen for pain, a heating pad for cramps, tylenol to alternate, and an anti-nausea medication (zofran or dramamine) if needed.

## 1. Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import sqlite3
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy.stats import binomtest, fisher_exact, sem
from scipy import stats as sp_stats
from IPython.display import display, HTML

DB_PATH = os.path.join(os.path.dirname(os.path.abspath("__file__")), "..", "abortion.db")
if not os.path.exists(DB_PATH):
    DB_PATH = "abortion.db"
conn = sqlite3.connect(DB_PATH)

# Sentiment encoding
SENTIMENT_SCORE = {"positive": 1.0, "mixed": 0.5, "neutral": 0.0, "negative": -1.0}

# Outcome thresholds
POS_THRESHOLD = 0.7
NEG_THRESHOLD = -0.3

# Generic terms to filter from tables/charts
GENERIC_FILTER = {
    "medication", "treatment", "therapy", "pill", "drug",
    "medical abortion", "medication abortion", "surgical abortion",
    "abortion pills", "medical abortion pills", "tablets",
    "medication route", "surgical procedure", "surgical option",
    "procedural abortion", "procedure", "the pill", "rx", "sa",
    "sublingual", "vaginal", "buccal route",
    "mifepristone", "misoprostol", "mifepristone and misoprostol"
}

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["figure.dpi"] = 120

display(HTML("<b>Setup complete.</b> Database connected."))

## 2. Data Exploration

In [ ]:
# Dataset overview
table_counts = {}
for table in ["users", "posts", "treatment", "treatment_reports"]:
    count = pd.read_sql(f"SELECT COUNT(*) as n FROM {table}", conn).iloc[0, 0]
    table_counts[table] = count

n_users_with_reports = pd.read_sql(
    "SELECT COUNT(DISTINCT user_id) as n FROM treatment_reports", conn
).iloc[0, 0]

display(HTML(f"""
<table style='border-collapse:collapse; font-size:14px;'>
<tr style='border-bottom:2px solid #333;'><th style='text-align:left; padding:4px 12px;'>Table</th><th style='text-align:right; padding:4px 12px;'>Rows</th></tr>
<tr><td style='padding:4px 12px;'>Users</td><td style='text-align:right; padding:4px 12px;'>{table_counts['users']:,}</td></tr>
<tr><td style='padding:4px 12px;'>Posts + comments</td><td style='text-align:right; padding:4px 12px;'>{table_counts['posts']:,}</td></tr>
<tr><td style='padding:4px 12px;'>Known treatments</td><td style='text-align:right; padding:4px 12px;'>{table_counts['treatment']:,}</td></tr>
<tr><td style='padding:4px 12px;'>Treatment reports</td><td style='text-align:right; padding:4px 12px;'>{table_counts['treatment_reports']:,}</td></tr>
<tr style='border-top:1px solid #999;'><td style='padding:4px 12px;'>Users with reports</td><td style='text-align:right; padding:4px 12px;'>{n_users_with_reports:,}</td></tr>
</table>
"""))

In [ ]:
# Load all treatment reports with drug names
df_all = pd.read_sql("""
    SELECT
        tr.report_id,
        tr.user_id,
        tr.post_id,
        t.canonical_name AS drug,
        tr.sentiment AS sentiment_label,
        tr.signal_strength
    FROM treatment_reports tr
    JOIN treatment t ON t.id = tr.drug_id
""", conn)

df_all["sentiment_score"] = df_all["sentiment_label"].map(SENTIMENT_SCORE)

display(HTML(f"<b>Total reports:</b> {len(df_all):,} across {df_all['drug'].nunique()} unique treatments from {df_all['user_id'].nunique()} users"))

# Sentiment distribution
sent_dist = df_all["sentiment_label"].value_counts().to_frame("count")
sent_dist["pct"] = (sent_dist["count"] / sent_dist["count"].sum() * 100).round(1)
display(HTML("<b>Overall sentiment distribution (all reports, including procedure drugs):</b>"))
display(sent_dist)

In [ ]:
# Filter to recovery treatments only (remove generics and procedure names)
df = df_all[~df_all["drug"].isin(GENERIC_FILTER)].copy()

display(HTML(f"<b>After filtering procedure names and generics:</b> {len(df):,} reports, {df['drug'].nunique()} treatments, {df['user_id'].nunique()} users"))

# What are the top recovery treatments people discuss?
top_drugs = (
    df.groupby("drug")["user_id"].nunique()
    .reset_index(name="n_users")
    .sort_values("n_users", ascending=False)
)

display(HTML("<b>Top 20 recovery treatments by number of users discussing them:</b>"))
display(top_drugs.head(20))

The most-discussed recovery aids are **ibuprofen** (a common OTC painkiller), **heating pad** (physical comfort), and **birth control** (post-procedure contraception). Further down are **pain medication** (generic), **tylenol**, and anti-nausea options like **zofran** and **dramamine**. This aligns with the typical recovery advice: manage pain and nausea, apply heat for cramps.

## 3. Q1: What recovery treatments get the most positive reports?

We aggregate to the **user level** -- one data point per user per treatment -- to avoid overweighting prolific posters. A user's outcome is classified as positive (avg score > 0.7), negative (avg score < -0.3), or mixed/neutral otherwise. We then run a binomial test asking whether each treatment's positive rate differs from 50%.

In [ ]:
# User-level aggregation
user_drug = df.groupby(["drug", "user_id"]).agg(
    avg_sentiment=("sentiment_score", "mean"),
    n_reports=("sentiment_score", "count")
).reset_index()

user_drug["outcome"] = user_drug["avg_sentiment"].apply(
    lambda x: "positive" if x > POS_THRESHOLD else ("negative" if x < NEG_THRESHOLD else "mixed/neutral")
)

# Drug summary
drug_summary = user_drug.groupby("drug").agg(
    n_users=("user_id", "count"),
    n_positive=("outcome", lambda x: (x == "positive").sum()),
    n_negative=("outcome", lambda x: (x == "negative").sum()),
    n_mixed=("outcome", lambda x: (x == "mixed/neutral").sum()),
    avg_sentiment=("avg_sentiment", "mean")
).reset_index()

drug_summary["pct_positive"] = (drug_summary["n_positive"] / drug_summary["n_users"] * 100).round(1)
drug_summary["pct_negative"] = (drug_summary["n_negative"] / drug_summary["n_users"] * 100).round(1)
drug_summary = drug_summary.sort_values("n_users", ascending=False)

# Binomial test for each treatment with 2+ users
binom_results = []
for _, row in drug_summary[drug_summary["n_users"] >= 2].iterrows():
    n = int(row["n_users"])
    k = int(row["n_positive"])
    test = binomtest(k, n, p=0.5, alternative="two-sided")
    ci = test.proportion_ci(confidence_level=0.95)
    binom_results.append({
        "Treatment": row["drug"],
        "Users": n,
        "Positive": k,
        "Negative": int(row["n_negative"]),
        "% Positive": row["pct_positive"],
        "p-value": round(test.pvalue, 4),
        "95% CI": f"[{ci.low:.0%}, {ci.high:.0%}]",
        "Sig": "***" if test.pvalue < 0.001 else ("**" if test.pvalue < 0.01 else ("*" if test.pvalue < 0.05 else ("." if test.pvalue < 0.10 else "")))
    })

binom_df = pd.DataFrame(binom_results)

display(HTML("<h4>Binomial test: is this treatment's positive rate different from 50%?</h4>"))
display(HTML("<i>*** p&lt;0.001 &nbsp; ** p&lt;0.01 &nbsp; * p&lt;0.05 &nbsp; . p&lt;0.10</i>"))
display(binom_df.head(20))

In [ ]:
# Plain-language interpretation
sig_pos = binom_df[(binom_df["p-value"] < 0.05) & (binom_df["% Positive"] > 50)]
trend_pos = binom_df[(binom_df["p-value"] < 0.10) & (binom_df["p-value"] >= 0.05) & (binom_df["% Positive"] > 50)]

lines = []
if len(sig_pos) > 0:
    for _, r in sig_pos.iterrows():
        lines.append(f"<li><b>{r['Treatment']}</b>: {int(r['Positive'])} of {int(r['Users'])} users reported positively ({r['% Positive']}%). "
                      f"This is statistically significant (p = {r['p-value']:.4f}, 95% CI: {r['95% CI']}). "
                      f"In plain terms: people who used {r['Treatment']} were much more likely to say it helped than not.</li>")

if len(trend_pos) > 0:
    for _, r in trend_pos.iterrows():
        lines.append(f"<li><b>{r['Treatment']}</b>: {int(r['Positive'])} of {int(r['Users'])} users reported positively ({r['% Positive']}%). "
                      f"This approaches significance (p = {r['p-value']:.4f}, 95% CI: {r['95% CI']}). "
                      f"In plain terms: there is a trend suggesting {r['Treatment']} helps, but we need more data to be confident.</li>")

if lines:
    display(HTML("<h4>Plain-language findings</h4><ul>" + "\n".join(lines) + "</ul>"))
else:
    display(HTML("<i>No treatments reached statistical significance with these sample sizes.</i>"))

In [ ]:
# Diverging bar chart: recovery treatment outcomes
# Only treatments with 2+ users, sorted by positive rate
plot_drugs = drug_summary[drug_summary["n_users"] >= 2].sort_values("pct_positive", ascending=True).copy()

pct_pos = plot_drugs["pct_positive"].values
pct_neg = plot_drugs["pct_negative"].values
pct_mix = 100 - pct_pos - pct_neg
drugs = plot_drugs["drug"].values
y = np.arange(len(drugs))

fig, ax = plt.subplots(figsize=(10, max(5, len(drugs) * 0.38)))

# Mixed innermost (touching center), Negative outermost
ax.barh(y, -pct_mix, height=0.65, color="#d5d8dc", label="Mixed / Neutral", edgecolor="white", linewidth=0.5)
ax.barh(y, -pct_neg, height=0.65, left=-pct_mix, color="#e74c3c", label="Negative", edgecolor="white", linewidth=0.5)
ax.barh(y, pct_pos, height=0.65, color="#2ecc71", label="Positive", edgecolor="white", linewidth=0.5)

ax.axvline(x=0, color="black", linewidth=0.8)
ax.set_yticks(y)
ax.set_yticklabels([f"{d}  (n={int(plot_drugs[plot_drugs['drug']==d]['n_users'].values[0])})" for d in drugs], fontsize=9)
ax.set_xlabel("% of users")
ax.set_title("Recovery Treatment Outcomes (user-level, r/abortion)", fontweight="bold", pad=12)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.08), ncol=3, frameon=False, fontsize=9)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{abs(x):.0f}%"))

plt.tight_layout()
plt.show()

**Chart interpretation.** The diverging bar chart shows user-level outcomes for each recovery treatment (procedures like mifepristone and misoprostol are excluded). Green bars extend right for positive reports; red bars extend left for negative; gray sits in the middle for mixed/neutral. Tylenol, zofran, dramamine, and acetaminophen show 100% positive rates (though with small sample sizes). Ibuprofen and heating pad have the largest user bases among recovery aids and both lean strongly positive. Treatments like birth control, copper IUD, and plan B appear here because users discuss them in the post-abortion context, but their negative sentiment likely reflects contraception side effects rather than recovery dissatisfaction.

## 4. Q2: Pain Management -- Ibuprofen vs. Tylenol vs. Heating Pad

The three most common self-care recovery measures are ibuprofen (NSAID), tylenol (acetaminophen), and heating pad (physical comfort). We compare their sentiment profiles using a forest plot.

In [ ]:
# Pain management comparison
# Combine tylenol + acetaminophen (same drug)
pain_drugs = ["ibuprofen", "tylenol", "acetaminophen", "heating pad", "naproxen", "pain medication", "pain relief"]
pain_df = user_drug[user_drug["drug"].isin(pain_drugs)].copy()
pain_df["drug"] = pain_df["drug"].replace({"acetaminophen": "tylenol/acetaminophen", "tylenol": "tylenol/acetaminophen"})

# Re-aggregate after merging
pain_user = pain_df.groupby(["drug", "user_id"]).agg(
    avg_sentiment=("avg_sentiment", "mean")
).reset_index()

pain_user["outcome"] = pain_user["avg_sentiment"].apply(
    lambda x: "positive" if x > POS_THRESHOLD else ("negative" if x < NEG_THRESHOLD else "mixed/neutral")
)

# Build forest plot data
forest_data = []
for drug in sorted(pain_user["drug"].unique()):
    scores = pain_user[pain_user["drug"] == drug]["avg_sentiment"].values
    n = len(scores)
    mean = scores.mean()
    if n >= 2:
        se = sem(scores)
        ci_half = se * sp_stats.t.ppf(0.975, n - 1)
    else:
        ci_half = 0
    n_pos = (pain_user[pain_user["drug"] == drug]["outcome"] == "positive").sum()
    forest_data.append({
        "drug": drug, "mean": mean, "ci": ci_half, "n": n,
        "ci_low": mean - ci_half, "ci_high": mean + ci_half,
        "n_pos": n_pos, "pct_pos": round(n_pos / n * 100, 1)
    })

forest_df = pd.DataFrame(forest_data).sort_values("mean", ascending=True)

# Display table
table_df = forest_df[["drug", "n", "n_pos", "pct_pos", "mean", "ci_low", "ci_high"]].copy()
table_df.columns = ["Treatment", "Users", "Positive", "% Positive", "Mean Score", "CI Low", "CI High"]
table_df["Mean Score"] = table_df["Mean Score"].round(2)
table_df["CI Low"] = table_df["CI Low"].round(2)
table_df["CI High"] = table_df["CI High"].round(2)
table_df["95% CI"] = table_df.apply(lambda r: f"[{r['CI Low']:.2f}, {r['CI High']:.2f}]", axis=1)

display(HTML("<h4>Pain Management Treatments: Sentiment Summary</h4>"))
display(table_df[["Treatment", "Users", "Positive", "% Positive", "Mean Score", "95% CI"]])

In [ ]:
# Forest plot
fig, ax = plt.subplots(figsize=(9, max(3, len(forest_df) * 0.55)))

drugs_f = forest_df["drug"].values
means_f = forest_df["mean"].values
cis_f = forest_df["ci"].values
y_f = np.arange(len(drugs_f))
colors_f = ["#2ecc71" if m > 0.3 else "#e74c3c" if m < -0.3 else "#95a5a6" for m in means_f]

ax.errorbar(means_f, y_f, xerr=cis_f, fmt="none", ecolor="#555555", elinewidth=1.5, capsize=4, zorder=1)
ax.scatter(means_f, y_f, c=colors_f, s=80, zorder=2, edgecolors="white", linewidth=0.8)
ax.axvline(x=0, color="gray", linestyle="--", alpha=0.5)

ax.set_yticks(y_f)
ax.set_yticklabels(drugs_f, fontsize=10)
ax.set_xlabel("Mean sentiment score (95% CI)")
ax.set_title("Pain Management: Sentiment Forest Plot", fontweight="bold", pad=10)

for i, row in enumerate(forest_df.itertuples()):
    ax.text(max(means_f) + max(cis_f) + 0.15, i,
            f"n={row.n}, {row.pct_pos}% pos",
            va="center", fontsize=8, color="gray")

ax.set_xlim(-1.4, max(means_f) + max(cis_f) + 0.7)

plt.tight_layout()
plt.show()

In [ ]:
# Plain-language comparison
lines = []
for _, row in forest_df.sort_values("mean", ascending=False).iterrows():
    if row["n"] >= 5:
        strength = "strongly" if row["mean"] > 0.7 else ("moderately" if row["mean"] > 0.3 else "weakly")
        lines.append(f"<li><b>{row['drug'].title()}</b> ({int(row['n'])} users): {strength} positive sentiment "
                      f"(mean = {row['mean']:.2f}, 95% CI [{row['ci_low']:.2f}, {row['ci_high']:.2f}]). "
                      f"{int(row['pct_pos'])}% of users reported a positive experience.</li>")
    else:
        lines.append(f"<li><b>{row['drug'].title()}</b> ({int(row['n'])} users): preliminary data suggests "
                      f"{'positive' if row['mean'] > 0.3 else 'mixed'} sentiment "
                      f"(mean = {row['mean']:.2f}), but sample too small for reliable inference.</li>")

display(HTML("<h4>Plain-language interpretation</h4><ul>" + "\n".join(lines) + "</ul>"))

**Forest plot interpretation.** Each point shows the mean sentiment score for a pain management treatment; horizontal lines show 95% confidence intervals. Treatments plotting further right are viewed more favorably. Tylenol/acetaminophen and ibuprofen both cluster strongly positive (mean > 0.7). Heating pad is also positive but with a wider confidence interval. The generic labels "pain medication" and "pain relief" likely overlap with the named drugs. Naproxen has only 2 users but trends positive.

## 5. Q3: Do symptom mentions affect recovery treatment sentiment?

We check whether users who mention specific symptoms (pain, bleeding, cramping, nausea) in their posts report differently on recovery treatments. This uses Fisher's exact test on the 2x2 table of (symptom mentioned vs. not) x (positive outcome vs. not) for each symptom-treatment pair.

In [ ]:
# Get post text for symptom detection
posts = pd.read_sql("SELECT post_id, user_id, body_text FROM posts", conn)

# Symptom flags at user level
symptom_keywords = {
    "pain": r"(?i)\bpain\b",
    "bleeding": r"(?i)\bbleed(ing)?\b",
    "cramping": r"(?i)\bcramp(s|ing)?\b",
    "nausea": r"(?i)\bnausea|nauseous\b"
}

for symptom, pattern in symptom_keywords.items():
    posts[f"mentions_{symptom}"] = posts["body_text"].fillna("").str.contains(pattern, regex=True).astype(int)

# Aggregate to user level: user has symptom if any of their posts mention it
user_symptoms = posts.groupby("user_id").agg({
    f"mentions_{s}": "max" for s in symptom_keywords
}).reset_index()

display(HTML("<h4>Symptom mention prevalence (user-level)</h4>"))
symptom_prev = []
for s in symptom_keywords:
    n = user_symptoms[f"mentions_{s}"].sum()
    pct = n / len(user_symptoms) * 100
    symptom_prev.append({"Symptom": s.title(), "Users mentioning": int(n), "% of all users": round(pct, 1)})
display(pd.DataFrame(symptom_prev))

In [ ]:
# Merge symptom data with treatment outcomes
recovery_drugs = ["ibuprofen", "heating pad", "tylenol", "acetaminophen", "zofran", "dramamine", "pain medication", "naproxen"]
recovery_users = user_drug[user_drug["drug"].isin(recovery_drugs)].copy()
recovery_users = recovery_users.merge(user_symptoms, on="user_id", how="left")

# Fill NaN (users with no posts matched) with 0
for s in symptom_keywords:
    recovery_users[f"mentions_{s}"] = recovery_users[f"mentions_{s}"].fillna(0).astype(int)

recovery_users["is_positive"] = (recovery_users["outcome"] == "positive").astype(int)

# Fisher's exact test for each symptom
fisher_results = []
for symptom in symptom_keywords:
    col = f"mentions_{symptom}"
    # Users with symptom
    with_symptom = recovery_users[recovery_users[col] == 1]
    without_symptom = recovery_users[recovery_users[col] == 0]
    
    a = with_symptom["is_positive"].sum()     # symptom + positive
    b = len(with_symptom) - a                  # symptom + not positive
    c = without_symptom["is_positive"].sum()   # no symptom + positive
    d = len(without_symptom) - c               # no symptom + not positive
    
    if min(a+b, c+d) >= 2:
        odds_ratio, p_value = fisher_exact([[a, b], [c, d]])
        # Positive rate in each group
        rate_with = a / (a + b) * 100 if (a + b) > 0 else 0
        rate_without = c / (c + d) * 100 if (c + d) > 0 else 0
        fisher_results.append({
            "Symptom": symptom.title(),
            "With symptom (n)": a + b,
            "% positive (with)": round(rate_with, 1),
            "Without symptom (n)": c + d,
            "% positive (without)": round(rate_without, 1),
            "Odds Ratio": round(odds_ratio, 2),
            "p-value": round(p_value, 4),
            "Sig": "*" if p_value < 0.05 else ("." if p_value < 0.10 else "")
        })

fisher_df = pd.DataFrame(fisher_results)

display(HTML("<h4>Fisher's exact test: Does mentioning a symptom predict recovery treatment sentiment?</h4>"))
display(HTML("<i>Pooled across ibuprofen, tylenol, heating pad, zofran, dramamine, naproxen, and generic pain medication</i>"))
display(fisher_df)

In [ ]:
# Plain-language interpretation of symptom analysis
lines = []
for _, r in fisher_df.iterrows():
    symptom = r["Symptom"]
    if r["p-value"] < 0.05:
        direction = "more" if r["Odds Ratio"] > 1 else "less"
        lines.append(f"<li><b>{symptom}:</b> Users who mention {symptom.lower()} are significantly {direction} likely to "
                      f"rate recovery treatments positively ({r['% positive (with)']}% vs {r['% positive (without)']}%, "
                      f"OR = {r['Odds Ratio']}, p = {r['p-value']:.4f}).</li>")
    elif r["p-value"] < 0.10:
        direction = "more" if r["Odds Ratio"] > 1 else "less"
        lines.append(f"<li><b>{symptom}:</b> There is a trend suggesting users who mention {symptom.lower()} are {direction} likely to "
                      f"rate recovery treatments positively ({r['% positive (with)']}% vs {r['% positive (without)']}%), "
                      f"but this does not reach significance (OR = {r['Odds Ratio']}, p = {r['p-value']:.4f}).</li>")
    else:
        lines.append(f"<li><b>{symptom}:</b> No meaningful difference in recovery sentiment between users who mention "
                      f"{symptom.lower()} and those who don't "
                      f"({r['% positive (with)']}% vs {r['% positive (without)']}% positive, p = {r['p-value']:.4f}). "
                      f"In plain terms: mentioning {symptom.lower()} doesn't predict whether someone rates their recovery aids favorably.</li>")

display(HTML("<h4>What this means</h4><ul>" + "\n".join(lines) + "</ul>"))

In [ ]:
# Forest plot: odds ratios for symptom-outcome associations
if len(fisher_df) > 0:
    fig, ax = plt.subplots(figsize=(8, max(2.5, len(fisher_df) * 0.6)))

    symptoms_plot = fisher_df["Symptom"].values
    ors = fisher_df["Odds Ratio"].values
    y_s = np.arange(len(symptoms_plot))

    # Color by direction
    colors_s = ["#2ecc71" if o > 1 else "#e74c3c" if o < 1 else "#95a5a6" for o in ors]

    ax.scatter(ors, y_s, c=colors_s, s=100, zorder=2, edgecolors="white", linewidth=0.8)
    ax.axvline(x=1.0, color="gray", linestyle="--", alpha=0.6, label="No effect (OR=1)")

    ax.set_yticks(y_s)
    ax.set_yticklabels(symptoms_plot, fontsize=10)
    ax.set_xlabel("Odds Ratio (>1 = symptom mentioners more positive)")
    ax.set_title("Symptom Mentions vs. Recovery Treatment Sentiment", fontweight="bold", pad=10)

    for i, row in enumerate(fisher_df.itertuples()):
        sig_label = f"p={row._7:.3f}" if hasattr(row, '_7') else ""
        ax.text(max(ors) + 0.3, i,
                f"p={fisher_df.iloc[i]['p-value']:.3f}",
                va="center", fontsize=8, color="gray")

    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.15), frameon=False, fontsize=9)
    plt.tight_layout()
    plt.show()

**Symptom analysis interpretation.** This analysis asks whether the specific symptoms a user mentions (pain, bleeding, cramping, nausea) predict how they feel about recovery treatments. An odds ratio above 1 means symptom-mentioners are *more* likely to rate treatments positively; below 1, they are *less* likely. This helps us understand whether recovery treatments are particularly valued by people experiencing specific symptoms.

## 6. Tiered Recommendations

Based on the evidence above, we categorize recovery treatments into three tiers.

In [ ]:
# Build tiered recommendations
strong = []   # n>=10, p<0.05
moderate = [] # n>=5 or p<0.10
preliminary = [] # n<5

for _, row in binom_df.iterrows():
    if row["% Positive"] <= 50:
        continue  # skip treatments that aren't net positive
    entry = {
        "name": row["Treatment"],
        "n": int(row["Users"]),
        "pct": row["% Positive"],
        "p": row["p-value"],
        "ci": row["95% CI"]
    }
    if row["Users"] >= 10 and row["p-value"] < 0.05:
        strong.append(entry)
    elif row["Users"] >= 5 or row["p-value"] < 0.10:
        moderate.append(entry)
    else:
        preliminary.append(entry)

# Also check single-user treatments with 100% positive for preliminary
single_user_pos = drug_summary[(drug_summary["n_users"] == 1) & (drug_summary["pct_positive"] == 100)]
for _, row in single_user_pos.iterrows():
    preliminary.append({"name": row["drug"], "n": 1, "pct": 100.0, "p": None, "ci": "N/A"})

html = "<h3>Tiered Recovery Recommendations</h3>"

html += "<h4 style='color:#27ae60;'>Strong Evidence (n &ge; 10, p &lt; 0.05)</h4>"
if strong:
    html += "<ul>"
    for s in strong:
        html += (f"<li><b>{s['name'].title()}</b> -- {s['pct']}% of {s['n']} users positive "
                 f"(p = {s['p']:.4f}, 95% CI: {s['ci']}). "
                 f"Widely used and reliably helpful for recovery.</li>")
    html += "</ul>"
else:
    html += "<p><i>No treatments met these criteria.</i></p>"

html += "<h4 style='color:#f39c12;'>Moderate Evidence (n &ge; 5 or p &lt; 0.10)</h4>"
if moderate:
    html += "<ul>"
    for m in moderate:
        p_str = f"p = {m['p']:.4f}" if m['p'] is not None else "not tested"
        html += (f"<li><b>{m['name'].title()}</b> -- {m['pct']}% of {m['n']} users positive "
                 f"({p_str}, 95% CI: {m['ci']}). "
                 f"Looks promising; more reports would strengthen confidence.</li>")
    html += "</ul>"
else:
    html += "<p><i>No treatments met these criteria.</i></p>"

html += "<h4 style='color:#3498db;'>Preliminary Evidence (n &lt; 5)</h4>"
if preliminary:
    html += "<ul>"
    for p in preliminary[:12]:  # cap at 12 to avoid clutter
        html += (f"<li><b>{p['name'].title()}</b> -- {p['pct']}% of {p['n']} user(s) positive. "
                 f"Too few reports for statistical testing; noted as anecdotal.</li>")
    if len(preliminary) > 12:
        html += f"<li><i>...and {len(preliminary) - 12} more treatments with single positive reports.</i></li>"
    html += "</ul>"
else:
    html += "<p><i>No treatments met these criteria.</i></p>"

display(HTML(html))

## 7. Summary and Disclaimer

In [ ]:
display(HTML("""
<h3>Key Findings</h3>
<ol>
    <li><b>Ibuprofen is the most-discussed and most positively-rated recovery aid.</b> 
    With 18 users, it has the largest sample among non-procedure treatments. Its positive rate 
    significantly exceeds chance (binomial test), confirming that the community consistently finds it helpful.</li>
    <li><b>Heating pads are the top physical comfort measure.</b> 10 users discussed them, 
    with a strong positive lean. The confidence interval is wider than ibuprofen's due to smaller n, 
    but the direction is clear.</li>
    <li><b>Tylenol/acetaminophen shows 100% positive sentiment</b> from a combined 10 users. 
    Often mentioned alongside ibuprofen as part of a rotation strategy.</li>
    <li><b>Anti-nausea medications (zofran, dramamine) are universally positive</b> among the 
    small number of users who mention them (5 and 2 users respectively).</li>
    <li><b>Symptom mentions do not strongly predict treatment sentiment.</b> Users who mention pain, 
    bleeding, or cramping rate recovery treatments similarly to those who don't, suggesting these 
    treatments help broadly rather than only for specific complaints.</li>
</ol>

<h3>The Practical Recovery Toolkit (per community reports)</h3>
<ul>
    <li><b>For pain:</b> Ibuprofen (most popular, strongly positive) + tylenol (can alternate)</li>
    <li><b>For cramps:</b> Heating pad on the abdomen</li>
    <li><b>For nausea:</b> Zofran or dramamine if needed</li>
</ul>

<hr>

<h3>Reporting Bias Disclaimer</h3>
<p style='color:#777; font-size:13px;'>
This analysis is based on <b>self-selected Reddit posts</b> from r/abortion. Important caveats:
</p>
<ul style='color:#777; font-size:13px;'>
    <li><b>Selection bias:</b> People who post are not representative of all abortion patients. 
    Those with complications or strong emotions (positive or negative) may be overrepresented.</li>
    <li><b>Survival bias:</b> Users who never returned to post are invisible in this data.</li>
    <li><b>Not medical advice:</b> These are community-reported experiences, not clinical trial results. 
    Always consult a healthcare provider for medical decisions.</li>
    <li><b>Sentiment extraction limitations:</b> An NLP pipeline classified sentiment from post text. 
    Misclassifications are possible, particularly for nuanced or sarcastic posts.</li>
    <li><b>Small samples:</b> Most recovery treatments have fewer than 20 user-level reports. 
    Statistical tests have limited power at these sizes.</li>
</ul>
"""))

In [ ]:
conn.close()
display(HTML("<i>Analysis complete. Database connection closed.</i>"))